# Rayleigh half-space phasor workflow

This example evaluates a linear Rayleigh wave in a homogeneous elastic half-space. The flat free surface is local `z=0`; the medium occupies `z<0`. Inputs are complex peak phasors using `q(t) = Re[P exp(-2πif(t−t_ref))]`. The spatial factor is `exp(+i k_h·(r_h−r0_h))`.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import units as u
from gwexpy.frequencyseries import FrequencySeries

import gragra
from gragra.analytic.seismic import rayleigh_halfspace
from gragra.constants import G_SI
from gragra.geometry import Frame, HalfSpace
from gragra.results import PhasorRepresentation, compare
from gragra.targets import TargetPoints

provenance = {
    "python_executable": str(Path(sys.executable).resolve()),
    "python_prefix": str(Path(sys.prefix).resolve()),
    "gragra_import_path": str(Path(gragra.__file__).resolve().parent),
    "gragra_version": __import__("importlib.metadata", fromlist=["version"]).version("gragra"),
    "gwexpy_version": __import__("importlib.metadata", fromlist=["version"]).version("gwexpy"),
}
from IPython.display import JSON, display
display(JSON(provenance))

## Define the physical input

The phase reference time is GPS `1400000000 s`. The displacement series has an irregular frequency grid and explicit complex vertical amplitudes. `PhasorRepresentation()` records the interpretation, while the frequency series carries meters and the frequency coordinates carry hertz.

In [ ]:
surface = Frame(name="surface")
halfspace = HalfSpace(surface)
rho = 2300 * u.kg / u.m**3
vp = 5800 * u.m / u.s
vs = 3200 * u.m / u.s
frequencies = np.array([1.0, 2.5, 5.0]) * u.Hz
xi0 = np.array([1 + 0.3j, 0.7 - 0.2j, -0.4 + 0.8j]) * u.nm
displacement = FrequencySeries(xi0, frequencies=frequencies, name="vertical displacement")
representation = PhasorRepresentation()
reference_time = (1_400_000_000, 0)
phase_origin = np.array([-4.0, 9.0, 0.0]) * u.m
direction = np.array([1.0, 2.0, 0.0])
targets = TargetPoints(
    np.array([[3, -1, 2], [3, -1, -2], [12, 9, 0.2], [-4, 7, -30]]) * u.m,
    ids=("A", "B", "C", "D"), frame=surface,
)
print(f"density: {rho}; vp: {vp}; vs: {vs}")
print(f"frequencies: {frequencies}; vertical displacement: {xi0}")
print(f"direction: {direction / np.linalg.norm(direction)}; phase origin: {phase_origin}")
print(f"GPS reference: {reference_time[0]} s + {reference_time[1]} ns")
print("targets (surface-frame x, y, z in m):")
display(pd.DataFrame(targets.positions.to_value(u.m), index=targets.ids, columns=["x", "y", "z"]))
model = rayleigh_halfspace(halfspace, rho, vp, vs, displacement, representation, reference_time, phase_origin, direction)
setup = model.at(targets)
report = setup.validate()
print("validation:", report)

## Inspect the geometry before running

Both projections show the surface, propagation direction, phase origin and every target. Confirm that A and C are above the surface and B and D are below it, and that all four target labels are visible.

In [ ]:
fig = plt.figure(figsize=(13, 6))
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
setup.plot_setup(view="3d", ax=ax3d)
ax3d.set_title("3D setup")
ax_xz = fig.add_subplot(1, 2, 2)
setup.plot_setup(view="xz", ax=ax_xz)
ax_xz.set_title("XZ projection")
handles, labels = ax3d.get_legend_handles_labels()
ax3d.get_legend().remove()
ax_xz.get_legend().remove()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8, framealpha=0.9, bbox_to_anchor=(0.5, 0.01))
fig.subplots_adjust(left=0.06, right=0.98, bottom=0.24, top=0.83, wspace=0.28)
plot_path = Path(
    os.environ.get(
        "GRAGRA_RAYLEIGH_PLOT",
        Path.home()
        / ".cache"
        / "gragra"
        / "rayleigh-phasor-workflow"
        / "rayleigh_phasor_workflow.png",
    )
)
plot_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()

### Visual checklist

- The gray plane is the `z=0` surface; the half-space is below it.
- The arrow lies in the surface and points along normalized `(1, 2, 0)`.
- The origin marker is at `(-4, 9, 0) m`.
- A and C are above, B and D are below, with labels matching the input table.

## Run and inspect native phasor results

`run()` returns potential, acceleration, and acceleration gradient as native GWexpy `FrequencySeriesMatrix` payloads. Matrix rows identify targets, columns identify components, and the frequency axis retains the supplied irregular Hz coordinates.

In [ ]:
result = setup.run()
evaluation = result.evaluation
context = result.context
assert context is evaluation.context
print("shared phasor context:", context.frame.name, context.representation)
print("normalized frequencies (Hz):", context.frequencies_hz)
print("potential binding shares context:", context.binding("potential").context is context)
for observable in ("potential", "acceleration", "gradient"):
    matrix = getattr(result, observable)
    print(f"{observable}: {type(matrix).__name__}, shape={matrix.shape}")
    print(f"rows={list(matrix.rows)}, columns={list(matrix.cols)}, frequencies={matrix.frequencies}")
    display(matrix)
print("representation descriptor:", context.representation)
print("parameters:")
display(evaluation.parameters)
print("formulas:")
display(evaluation.formulas)
print("references:")
display(evaluation.references)

## Re-express the phase origin

Move the phase origin by `Δr₀=(6,−2,0) m`. Its projection along the normalized propagation direction `(1,2,0)/√5` is `2/√5 m`, so the phase compensation is nontrivial. The next cell compares the compensated result with the original through public `compare()`, showing its norm and component-phase tables. It applies a stricter auxiliary norm check requiring differences to be at most `10⁻¹² S_q`; this implies the approved component-wise connection condition `|Δq| ≤ 10⁻¹² S_q + 10⁻¹² |q_reference|`. An origin-only shift without amplitude compensation is shown as a control and must exceed the auxiliary norm limit. The independent Green-function oracle remains in the test suite.

In [ ]:
delta_origin = np.array([6.0, -2.0, 0.0]) * u.m
k_h = (2 * np.pi * frequencies.to_value(u.Hz) / model.spec.rayleigh_speed_m_s)[:, None] * model.spec.direction[None, :]
k_dot_delta = k_h @ delta_origin.to_value(u.m)
display(pd.DataFrame({"frequency_Hz": frequencies.to_value(u.Hz), "k_dot_delta_rad": k_dot_delta}))

compensated_xi0 = xi0 * np.exp(1j * k_dot_delta)
shifted_displacement = FrequencySeries(
    compensated_xi0,
    frequencies=frequencies,
    name="compensated vertical displacement",
)
shifted = rayleigh_halfspace(
    halfspace,
    rho,
    vp,
    vs,
    shifted_displacement,
    representation,
    reference_time,
    phase_origin + delta_origin,
    direction,
).at(targets).run()
uncompensated = rayleigh_halfspace(
    halfspace,
    rho,
    vp,
    vs,
    displacement,
    representation,
    reference_time,
    phase_origin + delta_origin,
    direction,
).at(targets).run()

compensated_comparison = compare(reference=result, candidate=shifted)
display(compensated_comparison.summary)
display(compensated_comparison.phase_differences)

k = np.linalg.norm(k_h, axis=1)
xi0_magnitude = np.abs(xi0.to_value(u.m))
density_kg_m3 = rho.to_value(u.kg / u.m**3)
scales = {
    "potential": G_SI * density_kg_m3 * xi0_magnitude / k,
    "acceleration": G_SI * density_kg_m3 * xi0_magnitude,
    "gradient": G_SI * density_kg_m3 * xi0_magnitude * k,
}
units = {
    "potential": u.m**2 / u.s**2,
    "acceleration": u.m / u.s**2,
    "gradient": u.s**-2,
}
tolerance_rows = []
for observable, scale in scales.items():
    difference = compensated_comparison.summary[
        f"{observable}_absolute_difference"
    ].to_value(units[observable])
    allowed = 1e-12 * np.tile(scale, len(targets))
    ratio = float(np.max(difference / allowed))
    within_tolerance = bool(np.all(difference <= allowed))
    tolerance_rows.append(
        {
            "observable": observable,
            "max_difference_over_tolerance": ratio,
            "within_tolerance": within_tolerance,
        }
    )
    if not within_tolerance:
        raise AssertionError(f"{observable} exceeds the connection tolerance")
display(pd.DataFrame(tolerance_rows))

origin_only_comparison = compare(reference=result, candidate=uncompensated)
display(origin_only_comparison.summary)
display(origin_only_comparison.phase_differences)
origin_only_rows = []
for observable, scale in scales.items():
    difference = origin_only_comparison.summary[
        f"{observable}_absolute_difference"
    ].to_value(units[observable])
    allowed = 1e-12 * np.tile(scale, len(targets))
    exceeds_tolerance = bool(np.any(difference > allowed))
    origin_only_rows.append(
        {
            "observable": observable,
            "max_difference_over_tolerance": float(np.max(difference / allowed)),
            "exceeds_tolerance": exceeds_tolerance,
        }
    )
    if not exceeds_tolerance:
        raise AssertionError(f"{observable} origin-only control did not differ")
display(pd.DataFrame(origin_only_rows))


## Complex value, amplitude and phase

The following table reports every target/component/frequency entry. Amplitude is `|P|`; phase is `arg(P)` in degrees under the stated phasor convention.

In [ ]:
rows = []
for observable in ("potential", "acceleration", "gradient"):
    matrix = getattr(result, observable)
    values = np.asarray(matrix.value)
    for i, target_id in enumerate(matrix.rows):
        for j, component in enumerate(matrix.cols):
            for k, frequency in enumerate(matrix.frequencies.to_value(u.Hz)):
                value = values[i, j, k]
                rows.append({"observable": observable, "target": target_id, "component": component, "frequency_Hz": frequency, "phasor": value, "amplitude": abs(value), "phase_deg": np.angle(value, deg=True), "unit": str(matrix.units[i, j])})
phasor_table = pd.DataFrame(rows)
pd.set_option("display.max_rows", 100)
display(phasor_table)